# Churn Risk Prediction

This notebook predicts customer churn using a Random Forest classifier tracked with **MLflow**.

**Workflow:**
1. Load and explore the churn dataset
2. Preprocess: impute missing values + one-hot encode categoricals
3. Train Random Forest with `class_weight="balanced"` (16.8% churn rate)
4. Log model, params, and metrics to MLflow
5. Display evaluation results and top feature importances

In [0]:
import pandas as pd
import numpy as np

# Read from the bronze churn table and convert to a pandas DataFrame for exploration / downstream cells
df = spark.table("bronze_churn").toPandas()
print(f"Read {len(df)} rows from bronze_churn")

print(f"Dataset shape: {df.shape}")
print(f"\nChurn distribution:\n{df['Churn'].value_counts()}")
print(f"\nChurn rate: {df['Churn'].mean():.2%}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
df.head()

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Drop CustomerID — not a predictive feature
df = df.drop(columns=["CustomerID"])

# Identify column types
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols.remove("Churn")

print(f"Categorical columns: {categorical_cols}")
print(f"Numeric columns: {numeric_cols}")

# Build preprocessing pipeline
numeric_transformer = SimpleImputer(strategy="median")
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# Split data
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.2%}, Test churn rate: {y_test.mean():.2%}")

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

# Full pipeline: preprocessing + Random Forest
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",
    )),
])

# MLflow tracking
mlflow.set_experiment("/Users/saibharathparsam@outlook.com/churn_prediction")

with mlflow.start_run(run_name="churn_rf_baseline"):
    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    # Log params & metrics
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)

    # Log the full pipeline as an MLflow model
    mlflow.sklearn.log_model(model, "churn_rf_model")

    # Print results
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"ROC AUC:   {roc_auc:.4f}")
    print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

    # Feature importance
    feature_names = (
        numeric_cols
        + list(model.named_steps["preprocessor"]
            .named_transformers_["cat"]
            .named_steps["encoder"]
            .get_feature_names_out(categorical_cols))
    )
    importances = model.named_steps["classifier"].feature_importances_
    feat_df = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(10)
    )
    print(f"\nTop 10 Important Features:\n{feat_df.to_string(index=False)}")

## Key Findings & Decisions

### Confusion Matrix Explained

| | Predicted: No Churn | Predicted: Churn |
|---|---|---|
| **Actual: No Churn** | 935 (True Negatives) | 1 (False Positive) |
| **Actual: Churn** | 26 (False Negatives) | 164 (True Positives) |

- **True Negatives (935)** — Customers who stayed and the model correctly predicted they would stay. Largest block, reflecting the 83% non-churn majority.
- **True Positives (164)** — Customers who churned and the model caught them. This drives the recall of 86.3%.
- **False Positives (1)** — Only 1 customer was flagged as churn risk but actually stayed.
- **False Negatives (26)** — 26 churners slipped through undetected.

**Decision**: The model is conservative — it only predicts churn when very confident. To catch more at-risk customers, consider lowering the prediction threshold (e.g., predict churn if probability > 0.3 instead of 0.5) to reduce the 26 misses.

---

### Top 10 Feature Importances

| Rank | Feature | Importance | Interpretation |
|---|---|---|---|
| 1 | **Tenure** | 24.6% | Dominant predictor — short-tenure customers are much more likely to churn. Accounts for nearly a quarter of the model's decision-making. |
| 2 | **CashbackAmount** | 8.8% | Lower average cashback correlates with higher churn risk — customers not getting rewards are more likely to leave. |
| 3 | **WarehouseToHome** | 6.2% | Greater distance from warehouse to home increases churn — likely a proxy for delivery time frustration. |
| 4 | **Complain** | 6.1% | Customers who raised a complaint in the last month are at higher risk — a direct dissatisfaction signal. |
| 5 | **DaySinceLastOrder** | 5.8% | Longer gaps since the last order indicate disengagement and predict churn. |
| 6 | **NumberOfAddress** | 5.2% | More registered addresses correlate with churn — possibly indicating instability or frequent relocations. |
| 7 | **OrderAmountHikeFromlastYear** | 5.1% | Smaller year-over-year order increases signal declining engagement. |
| 8 | **SatisfactionScore** | 4.4% | Lower satisfaction scores naturally map to higher churn likelihood. |
| 9 | **NumberOfDeviceRegistered** | 3.0% | More registered devices slightly increases churn — may indicate tech-savvy customers who comparison-shop. |
| 10 | **OrderCount** | 2.8% | Fewer orders in the last month is a mild churn indicator. |

**Decision**: Tenure is the strongest signal by far — new customers are the most vulnerable. A retention program should prioritize short-tenure customers with low cashback, recent complaints, and long gaps since their last order for proactive outreach.

In [0]:
# Load Top 10 Important Features into silver model_findings table (overwrite)
from pyspark.sql import functions as F

# Extract feature names and importances from the trained model
feature_names = (
    numeric_cols
    + list(model.named_steps["preprocessor"]
        .named_transformers_["cat"]
        .named_steps["encoder"]
        .get_feature_names_out(categorical_cols))
)
importances = model.named_steps["classifier"].feature_importances_

feat_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
feat_df.insert(0, "rank", range(1, len(feat_df) + 1))

# Write to silver table in overwrite mode
spark.createDataFrame(feat_df).write.format("delta").mode("overwrite").saveAsTable("retail_demo.silver.model_findings")

# Add table comment for GenAI discoverability
spark.sql("""
    COMMENT ON TABLE retail_demo.silver.model_findings IS
    'Top 10 feature importances from the churn Random Forest model. Ranked by importance score. Queryable by Genie spaces and AI assistants for churn driver analysis.'
""")

print(f"Wrote {feat_df.shape[0]} rows to retail_demo.silver.model_findings")
display(spark.table("retail_demo.silver.model_findings"))